# Paper 1: theorem checks and Lean receipt

This notebook reads **no market data**. It does two things.

1. **Numerical checks.** Re-runs every numerical check cited in the paper and prints PASS or FAIL against the value
   the paper states (16+ checks, about 15-25 minutes). The optional slow cell adds the global-fibre search and the
   two-market worst-case table (about 30-60 minutes more).
2. **Lean.** Installs Lean, downloads the Mathlib cache, builds all modules of the formalization and runs
   `verify_formalization.py`, which writes `formal_verification_receipt.json` (about 15-30 minutes). Download the
   receipt at the end and commit it.

Repository: `https://github.com/Pratikn03/Fianince.git`, branch `claude/paper1-integrity-and-sujan-fixes-20261106`. Runtime: CPU is enough.

In [ ]:
!git clone --depth 1 -b claude/paper1-integrity-and-sujan-fixes-20261106 https://github.com/Pratikn03/Fianince.git repo
%cd repo
!pip -q install mpmath scipy numpy sympy
!git log --oneline -1

In [ ]:
# 1. Numerical checks (fast part)
!python3 research_extension/theorem_checks/run_all_checks.py

In [ ]:
# 1b. Optional slow part: global fibres for pooling and the two-market worst case
# !python3 research_extension/theorem_checks/run_all_checks.py --slow

In [ ]:
# 2. Lean: toolchain and Mathlib cache
!curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain none
import os
os.environ["PATH"] = os.path.expanduser("~/.elan/bin") + ":" + os.environ["PATH"]
%cd research_extension/current_manuscript_lean_20261008
!lake exe cache get

In [ ]:
# 3. Build every module and write the receipt
!lake build 2>&1 | tail -5
!python3 verify_formalization.py
!python3 -c "import json; r=json.load(open('formal_verification_receipt.json')); print({k: r[k] for k in list(r)[:12]})"

In [ ]:
# 4. Download the receipt and the check results
from google.colab import files
files.download('formal_verification_receipt.json')
files.download('../theorem_checks/run_all_checks_results.json')